
![DBAcademy](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/databricks_academy.png)

# Lecture - Working with the Rescued Data Column

## Overview

In this lecture, you will learn how the rescued data column (`_rescued_data`) captures mismatched or unparseable fields as JSON during data ingestion, preserving non-conforming input values in your Lakehouse tables instead of dropping them.

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Explain the purpose of the rescued data column** and how it preserves non-conforming data during ingestion
2. **Describe how schema mismatches are handled** when using `read_files()`, `spark.read`, or Auto Loader
3. **Interpret rescued data values** stored as JSON-formatted strings in the `_rescued_data` column

## A. Rescuing Malformed Rows on Ingestion

During data ingestion there are times when the input data does not match the schema in your table.

<div style="max-width:1100px; margin:0 auto; padding:14px 16px 6px; font-family:system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; color:#16313b; background:#fff;">

  <style>
    .resc-wrap{position:relative;}
    .top-callout{
      border:3px solid #ff5b43;
      padding:12px 16px 8px;
      text-align:center;
      font-size:26px;
      line-height:1.2;
      font-weight:400;
      margin:0 110px 20px 0;
      color:#142b34;
    }
    .top-callout b{font-weight:800;}

    /* simple elbow connector: right line + down line */
    .top-route{
      position:absolute;
      right:65px;
      top:44px;
      width:50px;
      height:150px;
      border-top:4px solid #ff5b43;
      border-right:4px solid #ff5b43;
      box-sizing:border-box;
    }

    .flow{
      display:flex;
      align-items:flex-start;
      justify-content:space-between;
      gap:16px;
      margin-top:4px;
    }

    .source-box{
      width:180px;
      height:320px;
      border:2px solid #1b5568;
      display:flex;
      flex-direction:column;
      align-items:center;
      padding-top:22px;
      box-sizing:border-box;
    }
    .source-box img{
      width:90px;
      height:70px;
      object-fit:contain;
      margin-bottom:12px;
      background:transparent;
      mix-blend-mode:multiply;
      filter:contrast(1.05) brightness(1.02);
    }
    .source-list{
      margin-top:4px;
      font-size:30px;
      line-height:1.7;
      text-align:center;
      font-weight:400;
    }

    .big-arrow{
      width:82px;
      height:30px;
      background:#3e93d9;
      clip-path:polygon(0 0, 70% 0, 70% -5%, 100% 50%, 70% 105%, 70% 100%, 0 100%);
      border:1.5px solid #165b8e;
      box-sizing:border-box;
      flex-shrink:0;
      margin-top:150px;
    }

    .ingestion{
      width:235px;
      height:300px;
      border:3px solid #4d97e3;
      border-radius:24px;
      padding:16px 18px;
      box-sizing:border-box;
    }
    .ingestion-title{
      font-size:28px;
      font-weight:700;
      color:#1d556a;
      text-align:center;
      margin-bottom:12px;
      line-height:1.1;
    }
    .ing-box{
      height:58px;
      display:flex;
      align-items:center;
      justify-content:center;
      font-size:20px;
      font-weight:700;
      color:#fff;
      margin-bottom:14px;
      box-sizing:border-box;
      border:1.5px solid rgba(0,0,0,.15);
    }
    .ing-blue{background:#1f78bf;}
    .ing-grey{background:#cfcfcf; color:#f7f7f7;}

    .bronze-col{
      width:230px;
      text-align:center;
      position:relative;
      padding-top:6px;
    }
    .bronze-col img{
      width:108px;
      height:80px;
      object-fit:contain;
      margin:0 auto 8px;
      display:block;
      background:transparent;
      mix-blend-mode:multiply;
      filter:contrast(1.05) brightness(1.02);
    }
    .bronze-label{
      font-size:38px;
      font-weight:800;
      color:#132730;
      line-height:1.02;
    }

    .rescue-col{width:300px; position:relative; margin-right:4px;}
    .rescue-table{
      border:1.5px solid #5d6f79;
      width:100%;
      background:#fff;
      box-sizing:border-box;
      margin-top:66px;
      position:relative;
    }
    .rescue-head{display:grid; grid-template-columns:30px 1fr;}
    .rescue-head .left{
      background:#1f78bf; color:#fff; padding:6px 6px; font-weight:700; font-size:20px; line-height:1;
      border-right:1.5px solid #5d6f79;
    }
    .rescue-head .right{
      background:#ff5b43; color:#fff; padding:5px 10px; font-weight:800; font-size:16px; line-height:1.15;
    }
    .rescue-row{
      display:grid;
      grid-template-columns:30px 1fr;
      border-top:1.5px solid #5d6f79;
      min-height:38px;
    }
    .rescue-row .left{
      border-right:1.5px solid #5d6f79;
      padding:8px 6px;
      font-size:18px;
      color:#3a4d57;
      line-height:1;
    }
    .rescue-row .right{
      padding:5px 10px;
      font-size:14px;
      line-height:1.2;
      font-weight:600;
      color:#20353d;
    }
    .red-x{position:absolute; right:86px; bottom:-22px; width:46px; height:46px;}
    .red-x:before,.red-x:after{
      content:""; position:absolute; left:20px; top:0; width:6px; height:46px; background:#ff3b30; border-radius:3px;
    }
    .red-x:before{transform:rotate(45deg);}
    .red-x:after{transform:rotate(-45deg);}
  </style>

  <div class="resc-wrap">
    <div class="top-callout">
      <b>read_files(), spark.read or Auto Loader</b> provides a <b>rescued data column</b> if the raw data does not match the schema
    </div>
    <div class="top-route"></div>
    <div class="flow">
      <div class="source-box">
        <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/cloud_icon.png" alt="Cloud icon">
        <div class="source-list">CSV<br>JSON<br>Parquet<br>etc.</div>
      </div>
      <div class="big-arrow"></div>
      <div class="ingestion">
        <div class="ingestion-title">Data Ingestion</div>
        <div class="ing-box ing-blue">CTAS</div>
        <div class="ing-box ing-grey">COPY INTO</div>
        <div class="ing-box ing-blue">AUTO LOADER</div>
      </div>
      <div class="big-arrow"></div>
      <div class="bronze-col">
        <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/bronze_table_icon.png" alt="Bronze table icon">
        <div class="bronze-label">Bronze Table</div>
      </div>
      <div class="big-arrow"></div>
      <div class="rescue-col">
        <div class="rescue-table">
          <div class="rescue-head">
            <div class="left">…</div>
            <div class="right">_rescued_data</div>
          </div>
          <div class="rescue-row">
            <div class="left">…</div>
            <div class="right">{"column": "< data >", "_file_path": "< file_path >"}</div>
          </div>
          <div class="rescue-row">
            <div class="left">…</div>
            <div class="right">{"column": "< data >", "_file_path": "< file_path >"}</div>
          </div>
          <div class="rescue-row">
            <div class="left">…</div>
            <div class="right">null</div>
          </div>
        </div>
        <div class="red-x"></div>
      </div>
    </div>
  </div>
</div>

##### EXPAND FOR ADDITIONAL NOTES
<details>

Ingestion techniques like `read_files()`, `spark.read`, or Auto Loader provide a rescued data column during ingestion:

- The rescued data column ensures that columns that do not match the schema are <strong>rescued instead of being dropped</strong>
- Mismatched values are stored as <strong>JSON-formatted strings</strong> in the <code>_rescued_data</code> column
- If a row has no schema mismatches, the <code>_rescued_data</code> column will be <code>null</code>
- This preserves all input data and prevents silent data loss

</details>

## B. Rescued Data Column Example

##### Click each step below to explore how to work with the Rescued Data Column.

<div style="width:100%;font-family:'Segoe UI',sans-serif;max-width:1100px;margin:0 auto;">

<style>
.rj-shell {
 display:flex;
 flex-direction:column;
 gap:16px;
}

/* Image */
.rj-figure-wrap{
 position:relative;
 width:720px;
 max-width:100%;
 margin:10px auto 0 auto;
 overflow:hidden;
 border-radius:8px;
}
.rj-figure-wrap img{
 width:100%;
 height:auto;
 display:block;
}
.rj-highlight{
 position:absolute;
 border:3px solid #FF3B30;
 border-radius:8px;
 box-shadow:0 0 0 9999px rgba(255,255,255,0.08);
 opacity:0;
 transition:opacity .25s ease;
 pointer-events:none;
}
.rj-highlight.active{ opacity:1; }

/* Highlight regions */
.rj-h1{ left:2%; top:3%; width:20%; height:31%; }     /* files */
.rj-h2{ left:1%; top:34%; width:23%; height:28%; }    /* users and cost */
.rj-h3{ left:39%; top:25%; width:20%; height:35%; }   /* STRING and BIGINT */
.rj-h4{ left:39%; top:40%; width:58%; height:7%; }    /* Peter + $100 main row */
.rj-h4b{ left:28%; top:77%; width:46%; height:17%; }  /* Peter + $100 extra box */
.rj-h5{ left:39%; top:46.5%; width:58%; height:7%; }  /* zebi + 300 main row */
.rj-h5b{ left:28%; top:61%; width:32%; height:15%; }  /* zebi + 300 extra box */

/* Horizontal steps */
.rj-list {
 display:flex;
 align-items:stretch;
 justify-content:center;
 gap:10px;
 flex-wrap:wrap;
}
.rj-trigger {
 display:flex;
 flex-direction:column;
 align-items:center;
 gap:8px;
 cursor:pointer;
 width:200px;
}
.rj-badge {
 width:22px;
 height:22px;
 border-radius:50%;
 display:flex;
 align-items:center;
 justify-content:center;
 font-size:16px;
 font-weight:800;
 color:white;
 border:3px solid white;
 box-shadow:0 2px 10px rgba(0,0,0,0.15);
 transition:transform 0.2s, box-shadow 0.2s;
}
.rj-trigger:hover .rj-badge {
 transform:scale(1.08);
 box-shadow:0 6px 18px rgba(0,0,0,0.2);
}
.rj-label-box {
 width:100%;
 background:#F9F7F4;
 border-radius:8px;
 border:1.5px solid #e8e5e0;
 padding:10px 12px;
 transition:border-color 0.2s, box-shadow 0.2s;
 box-sizing:border-box;
 min-height:82px;
}
.rj-trigger.active .rj-label-box {
 border-color:var(--step-color);
 box-shadow:0 2px 10px rgba(0,0,0,0.10);
 background:#fff;
}
.rj-trigger:hover .rj-label-box {
 box-shadow:0 2px 8px rgba(0,0,0,0.08);
}
.rj-label-top {
 display:flex;
 align-items:center;
 gap:7px;
 justify-content:center;
}
.rj-dot {
 width:9px;
 height:9px;
 border-radius:50%;
 background:var(--step-color);
 flex-shrink:0;
}
.rj-title {
 font-size:12pt;
 font-weight:700;
 color:#1b3139;
 line-height:1.25;
 text-align:center;
}
.rj-sub {
 font-size:9pt;
 color:#7a7974;
 margin-top:4px;
 line-height:1.35;
 text-align:center;
}
.rj-chevron {
 font-size:11px;
 color:#ccc;
 transition:transform 0.25s, color 0.2s;
 flex-shrink:0;
}
.rj-trigger.active .rj-chevron {
 transform:rotate(90deg);
 color:var(--step-color);
}

.rj-panels { width:100%; }

.rj-panel {
 display:none;
 background:#F9F7F4;
 border-radius:12px;
 border:1.5px solid #e8e5e0;
 padding:20px 22px;
 animation:rjIn 0.28s ease;
}
.rj-panel.active {
 display:block;
 border-color:var(--panel-color);
 box-shadow:0 4px 20px rgba(0,0,0,0.10);
}
@keyframes rjIn {
 from { opacity:0; transform:translateY(-10px); }
 to { opacity:1; transform:translateY(0); }
}

.rj-panel-title {
 display:flex;
 align-items:center;
 gap:10px;
 margin-bottom:14px;
 padding-bottom:10px;
 border-bottom:1.5px solid #eeede9;
}
.rj-panel-dot {
 width:11px;
 height:11px;
 border-radius:50%;
 background:var(--panel-color);
 flex-shrink:0;
}
.rj-panel-title h4 {
 font-size:13pt;
 font-weight:800;
 color:#1b3139;
 margin:0;
}
.rj-panel-title span {
 font-size:11pt;
 color:#7a7974;
 margin-left:4px;
}
.rj-desc {
 font-size:11pt;
 color:#3a3a3a;
 line-height:1.75;
 margin:0 0 12px 0;
}
.rj-desc strong { color:#1b3139; }

.tbl-wrap { overflow-x:auto; margin-top:12px; }
.rj-table { width:100%; border-collapse:collapse; font-size:9.5pt; }
.rj-table th { padding:7px 11px; text-align:left; color:white; font-weight:700; font-size:9pt; }
.rj-table td { padding:7px 11px; color:#1b3139; border-bottom:1px solid #f0edea; font-size:9pt; }
.rj-table tr:last-child td { border-bottom:none; color:#aaa; }

.th-blue { background:#2574B5; }
.th-orange { background:#E05A2B; }
.td-rescued { background:rgba(224,90,43,0.07); font-size:8.5pt; font-family:monospace; color:#c0400a; }
.td-null { color:#aaa !important; font-style:italic; }
.td-ok { background:rgba(2,163,111,0.06); }

.callout {
 margin-top:12px;
 padding:11px 15px;
 border-radius:8px;
 font-size:10.5pt;
 line-height:1.65;
}
.callout-orange { background:rgba(224,90,43,0.09); border-left:4px solid #E05A2B; }
.callout-green { background:rgba(2,163,111,0.08); border-left:4px solid #02A36F; }

.tag-row { display:flex; gap:8px; flex-wrap:wrap; margin-top:10px; }
.tag { padding:3px 11px; border-radius:999px; font-size:9pt; font-weight:600; color:white; }
</style>

<div class="rj-shell">
 <!-- Image -->
 <div class="rj-figure-wrap">
  <img
   src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/lecture_rescued_data/rescued_data_example.png"
   alt="Rescued Data Column">
  <div class="rj-highlight rj-h1" id="rh1"></div>
  <div class="rj-highlight rj-h2" id="rh2"></div>
  <div class="rj-highlight rj-h3" id="rh3"></div>
  <div class="rj-highlight rj-h4" id="rh4"></div>
  <div class="rj-highlight rj-h4b" id="rh4b"></div>
  <div class="rj-highlight rj-h5" id="rh5"></div>
  <div class="rj-highlight rj-h5b" id="rh5b"></div>
 </div>
 <!-- Horizontal steps -->
 <div class="rj-list">
  <div class="rj-trigger active" style="--step-color:#2574B5;" onclick="rjSwitch(this,'rp1','rh1')">
   <div class="rj-badge" style="background:#2574B5;">1</div>
   <div class="rj-label-box">
    <div class="rj-label-top">
     <div class="rj-dot"></div>
     <div class="rj-title">Files</div>
     <span class="rj-chevron">▶</span>
    </div>
    <div class="rj-sub">raw files in cloud storage</div>
   </div>
  </div>
  <div class="rj-trigger" style="--step-color:#E05A2B;" onclick="rjSwitch(this,'rp2','rh2')">
   <div class="rj-badge" style="background:#E05A2B;">2</div>
   <div class="rj-label-box">
    <div class="rj-label-top">
     <div class="rj-dot"></div>
     <div class="rj-title">Users and Cost</div>
     <span class="rj-chevron">▶</span>
    </div>
    <div class="rj-sub">source columns in the raw files</div>
   </div>
  </div>
  <div class="rj-trigger" style="--step-color:#02A36F;" onclick="rjSwitch(this,'rp3','rh3')">
   <div class="rj-badge" style="background:#02A36F;">3</div>
   <div class="rj-label-box">
    <div class="rj-label-top">
     <div class="rj-dot"></div>
     <div class="rj-title">STRING and BIGINT</div>
     <span class="rj-chevron">▶</span>
    </div>
    <div class="rj-sub">expected types during ingestion</div>
   </div>
  </div>
  <div class="rj-trigger" style="--step-color:#1C3037;" onclick="rjSwitch(this,'rp4','rh4')">
   <div class="rj-badge" style="background:#1C3037;">4</div>
   <div class="rj-label-box">
    <div class="rj-label-top">
     <div class="rj-dot"></div>
     <div class="rj-title">"Peter" and "$100"</div>
     <span class="rj-chevron">▶</span>
    </div>
    <div class="rj-sub">captured in _rescued_data</div>
   </div>
  </div>
  <div class="rj-trigger" style="--step-color:#618794;" onclick="rjSwitch(this,'rp5','rh5')">
   <div class="rj-badge" style="background:#618794;">5</div>
   <div class="rj-label-box">
    <div class="rj-label-top">
     <div class="rj-dot"></div>
     <div class="rj-title">"zebi" and "300"</div>
     <span class="rj-chevron">▶</span>
    </div>
    <div class="rj-sub">read without issues</div>
   </div>
  </div>
 </div>
 <!-- Panels -->
 <div class="rj-panels">
  <div class="rj-panel active" id="rp1" style="--panel-color:#2574B5;">
   <div class="rj-panel-title">
    <div class="rj-panel-dot"></div>
    <h4>files</h4>
    <span>raw files in cloud storage</span>
   </div>
   <p class="rj-desc">
    Suppose your cloud storage location contains a set of raw files. These could be CSV, TXT, JSON, or other formats.
   </p>
   <div class="tag-row">
    <span class="tag" style="background:#2574B5;">CSV</span>
    <span class="tag" style="background:#2574B5;">TXT</span>
    <span class="tag" style="background:#2574B5;">JSON</span>
   </div>
  </div>
  <div class="rj-panel" id="rp2" style="--panel-color:#E05A2B;">
   <div class="rj-panel-title">
    <div class="rj-panel-dot"></div>
    <h4>users and cost</h4>
    <span>columns in the raw files</span>
   </div>
   <p class="rj-desc">
    The raw files contain a <strong>users</strong> column and a <strong>cost</strong> column.
   </p>
   <div class="tbl-wrap">
    <table class="rj-table" style="max-width:320px;">
     <thead>
      <tr><th class="th-blue">users</th><th class="th-blue">cost</th></tr>
     </thead>
     <tbody>
      <tr><td>Peter</td><td>$100</td></tr>
      <tr><td>zebi</td><td>300</td></tr>
     </tbody>
    </table>
   </div>
  </div>
  <div class="rj-panel" id="rp3" style="--panel-color:#02A36F;">
   <div class="rj-panel-title">
    <div class="rj-panel-dot"></div>
    <h4>STRING and BIGINT</h4>
    <span>expected types during ingestion</span>
   </div>
   <p class="rj-desc">
    When ingesting this data, the <strong>users</strong> column must be read into the table as a <strong>STRING</strong>, and the <strong>cost</strong> column must be read as a <strong>BIGINT</strong>.
   </p>
   <div class="tbl-wrap">
    <table class="rj-table" style="max-width:360px;">
     <thead>
      <tr><th class="th-blue">Column</th><th class="th-blue">Expected Type</th></tr>
     </thead>
     <tbody>
      <tr><td>users</td><td>STRING</td></tr>
      <tr><td>cost</td><td>BIGINT</td></tr>
     </tbody>
    </table>
   </div>
  </div>
  <div class="rj-panel" id="rp4" style="--panel-color:#1C3037;">
   <div class="rj-panel-title">
    <div class="rj-panel-dot"></div>
    <h4>First row: "Peter" and "$100"</h4>
    <span>_rescued_data is populated</span>
   </div>
   <p class="rj-desc">
    In the first row of data, the value <strong>"Peter"</strong> will be read into the <strong>users</strong> column of the bronze table correctly. However, since the <strong>cost</strong> column contains a string like <strong>$100</strong>, it does not match the expected <strong>BIGINT</strong> type.
   </p>
   <div class="callout callout-orange">
    As a result, this value will not be inserted into the cost column. Instead, it will be captured in the <strong>_rescued_data</strong> column, stored as a <strong>JSON-formatted string</strong>.
   </div>
   <div class="tbl-wrap">
    <table class="rj-table">
     <thead>
      <tr>
       <th class="th-blue">users</th>
       <th class="th-blue">cost</th>
       <th class="th-orange">_rescued_data</th>
      </tr>
     </thead>
     <tbody>
      <tr>
       <td>Peter</td>
       <td class="td-null">null</td>
       <td class="td-rescued">{"cost": "$100", "_file_path": "&lt;file_path&gt;"}</td>
      </tr>
     </tbody>
    </table>
   </div>
  </div>
  <div class="rj-panel" id="rp5" style="--panel-color:#618794;">
   <div class="rj-panel-title">
    <div class="rj-panel-dot"></div>
    <h4>Second row: "zebi" and 300</h4>
    <span>_rescued_data remains empty</span>
   </div>
   <p class="rj-desc">
    In the second row, both values are valid: the value <strong>"zebi"</strong> is a <strong>STRING</strong>, and <strong>300</strong> is a <strong>BIGINT</strong>.
   </p>
   <div class="callout callout-green">
    Therefore, this row will be read into the bronze table without issues, and the <strong>_rescued_data</strong> column will remain empty for that row.
   </div>
   <div class="tbl-wrap">
    <table class="rj-table">
     <thead>
      <tr>
       <th class="th-blue">users</th>
       <th class="th-blue">cost</th>
       <th class="th-orange">_rescued_data</th>
      </tr>
     </thead>
     <tbody>
      <tr>
       <td>zebi</td>
       <td class="td-ok">300</td>
       <td class="td-null">null</td>
      </tr>
     </tbody>
    </table>
   </div>
  </div>
 </div>
</div>

<script>
function rjSwitch(triggerEl, panelId, highlightId) {
  var allTriggers = document.querySelectorAll('.rj-trigger');
  var allPanels = document.querySelectorAll('.rj-panel');
  var allHighlights = document.querySelectorAll('.rj-highlight');
  var isAlreadyActive = triggerEl.classList.contains('active');

  allTriggers.forEach(function(t){ t.classList.remove('active'); });
  allPanels.forEach(function(p){ p.classList.remove('active'); });
  allHighlights.forEach(function(h){ h.classList.remove('active'); });

  if (!isAlreadyActive) {
    triggerEl.classList.add('active');
    document.getElementById(panelId).classList.add('active');
    document.getElementById(highlightId).classList.add('active');

    if (highlightId === 'rh4') {
      document.getElementById('rh4b').classList.add('active');
    }
    if (highlightId === 'rh5') {
      document.getElementById('rh5b').classList.add('active');
    }
  }
}
document.getElementById('rh1').classList.add('active');
</script>
</div>

##### EXPAND FOR ADDITIONAL NOTES
<details>

For example, suppose your cloud storage location contains a set of raw files (these could be CSV, TXT, JSON, or other formats) and it contains a users and cost column.

When ingesting this data, the users column must be read into the table as a STRING, and the cost column must be read as a BIGINT.

In the first row of data, the value "Peter" will be read into the users column of the bronze table correctly. However, since the cost column contains a string like $100, it does not match the expected BIGINT type. As a result, this value will not be inserted into the cost column. Instead, it will be captured in the <code>_rescued_data</code> column, stored as a JSON-formatted string.

In the second row, both values are valid: the value "zebi" is a STRING, and 300 is a BIGINT. Therefore, this row will be read into the bronze table without issues, and the <code>_rescued_data</code> column will remain empty for that row.


</details>

## C. Conclusion

In this lecture, you learned how the rescued data column works during data ingestion:

- When input data does not match the expected schema, **mismatched values are captured in the `_rescued_data` column** as JSON-formatted strings instead of being dropped.
- The rescued data column is available when using **`read_files()`**, **`spark.read`**, or **Auto Loader**.
- Values that match the schema are ingested normally, and `_rescued_data` is `null` for those rows.
- This feature prevents silent data loss and allows you to inspect and address schema mismatches after ingestion.

### Next Steps

In the next section, you will work hands-on with the rescued data column to handle schema mismatches during ingestion.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/>
<a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> |
<a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> |
<a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>